In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = "retail_demo"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}"
)

RUN_TIMESTAMP = spark.sql(
    "SELECT current_timestamp() AS run_timestamp"
).first()["run_timestamp"]

print("Dimension processing started at:", RUN_TIMESTAMP)

In [0]:
def prepare_dimension_source(
    source_table,
    business_key,
    tracked_columns
):

    source_df = spark.table(source_table)

    # Remove rows with missing business key
    source_df = source_df.filter(
        F.col(business_key).isNotNull() &
        (F.length(F.trim(F.col(business_key))) > 0)
    )

    # Keep latest record for each business key
    if "_bronze_load_timestamp" in source_df.columns:

        dedup_window = (
            Window
            .partitionBy(business_key)
            .orderBy(
                F.col(
                    "_bronze_load_timestamp"
                ).desc_nulls_last()
            )
        )

        source_df = (
            source_df
            .withColumn(
                "_row_number",
                F.row_number().over(dedup_window)
            )
            .filter(
                F.col("_row_number") == 1
            )
            .drop("_row_number")
        )

    else:

        source_df = source_df.dropDuplicates(
            [business_key]
        )

    # Build one combined string from tracked columns
    combined_expression = None

    for column_name in tracked_columns:

        column_expression = F.coalesce(
            F.col(column_name).cast("string"),
            F.lit("<NULL>")
        )

        if combined_expression is None:

            combined_expression = column_expression

        else:

            combined_expression = F.concat(
                combined_expression,
                F.lit("||"),
                column_expression
            )

    # Generate hash for change detection
    source_df = source_df.withColumn(
        "attribute_hash",
        F.sha2(
            combined_expression,
            256
        )
    )

    return source_df

In [0]:
def apply_scd_type_1(
    source_df,
    target_table,
    business_key
):

    if not spark.catalog.tableExists(target_table):

        initial_df = (
            source_df
            .withColumn(
                "_silver_created_timestamp",
                F.lit(RUN_TIMESTAMP)
            )
            .withColumn(
                "_silver_updated_timestamp",
                F.lit(RUN_TIMESTAMP)
            )
        )

        (
            initial_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
        )

        print("Created SCD Type 1 table:", target_table)

        return

    target_delta = DeltaTable.forName(
        spark,
        target_table
    )

    source_for_merge = (
        source_df
        .withColumn(
            "_silver_updated_timestamp",
            F.lit(RUN_TIMESTAMP)
        )
    )

    source_columns = source_df.columns

    update_mapping = {
        column_name: f"source.{column_name}"
        for column_name in source_columns
        if column_name != business_key
    }

    update_mapping[
        "_silver_updated_timestamp"
    ] = "source._silver_updated_timestamp"

    insert_mapping = {
        column_name: f"source.{column_name}"
        for column_name in source_columns
    }

    insert_mapping[
        "_silver_created_timestamp"
    ] = "source._silver_updated_timestamp"

    insert_mapping[
        "_silver_updated_timestamp"
    ] = "source._silver_updated_timestamp"

    (
        target_delta.alias("target")
        .merge(
            source_for_merge.alias("source"),
            f"target.{business_key} = source.{business_key}"
        )
        .whenMatchedUpdate(
            condition=(
                "target.attribute_hash <> "
                "source.attribute_hash"
            ),
            set=update_mapping
        )
        .whenNotMatchedInsert(
            values=insert_mapping
        )
        .execute()
    )

    print("SCD Type 1 completed:", target_table)

In [0]:
def apply_scd_type_2(
    source_df,
    target_table,
    business_key
):

    far_future = F.to_timestamp(
        F.lit("9999-12-31 23:59:59")
    )

    if not spark.catalog.tableExists(target_table):

        initial_df = (
            source_df
            .withColumn(
                "effective_from",
                F.lit(RUN_TIMESTAMP)
            )
            .withColumn(
                "effective_to",
                far_future
            )
            .withColumn(
                "is_current",
                F.lit(True)
            )
            .withColumn(
                "_silver_created_timestamp",
                F.lit(RUN_TIMESTAMP)
            )
            .withColumn(
                "_silver_updated_timestamp",
                F.lit(RUN_TIMESTAMP)
            )
        )

        (
            initial_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
        )

        print("Created SCD Type 2 table:", target_table)

        return

    target_current = (
        spark.table(target_table)
        .filter(
            F.col("is_current") == True
        )
        .select(
            business_key,
            F.col("attribute_hash").alias(
                "target_attribute_hash"
            )
        )
    )

    comparison_df = (
        source_df
        .join(
            target_current,
            on=business_key,
            how="left"
        )
    )

    changed_records = (
        comparison_df
        .filter(
            F.col("target_attribute_hash").isNotNull() &
            (
                F.col("attribute_hash") !=
                F.col("target_attribute_hash")
            )
        )
        .drop("target_attribute_hash")
    )

    new_records = (
        comparison_df
        .filter(
            F.col("target_attribute_hash").isNull()
        )
        .drop("target_attribute_hash")
    )

    changed_count = changed_records.count()
    new_count = new_records.count()

    if changed_count > 0:

        changed_keys = changed_records.select(
            business_key
        )

        changed_keys = changed_keys.withColumn(
            "effective_to",
            F.lit(RUN_TIMESTAMP)
        )

        target_delta = DeltaTable.forName(
            spark,
            target_table
        )

        (
            target_delta.alias("target")
            .merge(
                changed_keys.alias("source"),
                (
                    f"target.{business_key} = "
                    f"source.{business_key} "
                    "AND target.is_current = true"
                )
            )
            .whenMatchedUpdate(
                set={
                    "effective_to":
                        "source.effective_to",
                    "is_current":
                        "false",
                    "_silver_updated_timestamp":
                        "source.effective_to"
                }
            )
            .execute()
        )

    records_to_insert = (
        new_records
        .unionByName(
            changed_records,
            allowMissingColumns=True
        )
    )

    if new_count > 0 or changed_count > 0:

        records_to_insert = (
            records_to_insert
            .withColumn(
                "effective_from",
                F.lit(RUN_TIMESTAMP)
            )
            .withColumn(
                "effective_to",
                far_future
            )
            .withColumn(
                "is_current",
                F.lit(True)
            )
            .withColumn(
                "_silver_created_timestamp",
                F.lit(RUN_TIMESTAMP)
            )
            .withColumn(
                "_silver_updated_timestamp",
                F.lit(RUN_TIMESTAMP)
            )
        )

        (
            records_to_insert.write
            .format("delta")
            .mode("append")
            .saveAsTable(target_table)
        )

    print("SCD Type 2 completed:", target_table)
    print("New records:", new_count)
    print("Changed records:", changed_count)

In [0]:
CUSTOMER_SOURCE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.dim_customer"
)

CUSTOMER_TARGET = (
    f"{CATALOG}.{SILVER_SCHEMA}.dim_customer"
)

customer_tracked_columns = [
    "customer_name",
    "email",
    "gender",
    "age",
    "city",
    "state",
    "region",
    "segment",
    "signup_date",
    "loyalty_member",
    "loyalty_points"
]

customer_source_df = prepare_dimension_source(
    source_table=CUSTOMER_SOURCE,
    business_key="customer_id",
    tracked_columns=customer_tracked_columns
)

display(
    customer_source_df.limit(10)
)

In [0]:
apply_scd_type_2(
    source_df=customer_source_df,
    target_table=CUSTOMER_TARGET,
    business_key="customer_id"
)

In [0]:
PRODUCT_SOURCE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.dim_product"
)

PRODUCT_TARGET = (
    f"{CATALOG}.{SILVER_SCHEMA}.dim_product"
)

product_tracked_columns = [
    "product_name",
    "category",
    "subcategory",
    "brand",
    "unit_cost",
    "list_price",
    "weight_kg",
    "product_status",
    "tax_code"
]

product_source_df = prepare_dimension_source(
    source_table=PRODUCT_SOURCE,
    business_key="product_id",
    tracked_columns=product_tracked_columns
)

display(
    product_source_df.limit(10)
)

In [0]:
apply_scd_type_2(
    source_df=product_source_df,
    target_table=PRODUCT_TARGET,
    business_key="product_id"
)

In [0]:
STORE_SOURCE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.dim_store"
)

STORE_TARGET = (
    f"{CATALOG}.{SILVER_SCHEMA}.dim_store"
)

store_tracked_columns = [
    "store_name",
    "city",
    "state",
    "region",
    "store_format",
    "open_date",
    "employee_count"
]

store_source_df = prepare_dimension_source(
    source_table=STORE_SOURCE,
    business_key="store_id",
    tracked_columns=store_tracked_columns
)

display(
    store_source_df.limit(10)
)

In [0]:
apply_scd_type_1(
    source_df=store_source_df,
    target_table=STORE_TARGET,
    business_key="store_id"
)

In [0]:
CAMPAIGN_SOURCE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.dim_campaign"
)

CAMPAIGN_TARGET = (
    f"{CATALOG}.{SILVER_SCHEMA}.dim_campaign"
)

campaign_tracked_columns = [
    "campaign_name",
    "start_date",
    "end_date",
    "marketing_channel",
    "budget_inr",
    "objective"
]

campaign_source_df = prepare_dimension_source(
    source_table=CAMPAIGN_SOURCE,
    business_key="campaign_id",
    tracked_columns=campaign_tracked_columns
)

display(
    campaign_source_df.limit(10)
)

In [0]:
apply_scd_type_1(
    source_df=campaign_source_df,
    target_table=CAMPAIGN_TARGET,
    business_key="campaign_id"
)

In [0]:
silver_dimension_tables = [
    "dim_customer",
    "dim_product",
    "dim_store",
    "dim_campaign"
]

for table_name in silver_dimension_tables:

    full_table_name = (
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    )

    row_count = spark.table(
        full_table_name
    ).count()

    print(
        f"{full_table_name}: {row_count:,} records"
    )

In [0]:
customer_current_duplicates = (
    spark.table(CUSTOMER_TARGET)
    .filter(
        F.col("is_current") == True
    )
    .groupBy("customer_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

product_current_duplicates = (
    spark.table(PRODUCT_TARGET)
    .filter(
        F.col("is_current") == True
    )
    .groupBy("product_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

print(
    "Customers with multiple current versions:",
    customer_current_duplicates
)

print(
    "Products with multiple current versions:",
    product_current_duplicates
)

assert customer_current_duplicates == 0
assert product_current_duplicates == 0

print("SCD Type 2 validation passed.")